In [2]:
from rdflib import Graph, URIRef, Literal
import csv
from sodapy import Socrata
from rdflib import Graph, URIRef, Literal
import csv,os
from sodapy import Socrata
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS




In [70]:
def getCimDatasets():
    cim_url_query = "data.colorado.gov"
    allDatasets = []
    cimDatasets = {}
    
    # Connect to the Socrata API
    with Socrata(cim_url_query, None) as client:
        datasets = client.datasets()
        for dataset in datasets:
            allDatasets.append(dataset)
            if dataset['owner']['display_name'] == 'Colorado Information Marketplace' or dataset['owner']['display_name'] == "Business Intelligence Center of CO":
                title=dataset["resource"]["name"]
                w4x4=dataset["resource"]["id"]

                cimDatasets[title]=dataset
    
    return cimDatasets
dfCim = getCimDatasets()

In [53]:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
sentences={}
words={}
domains = {}
tags = {}
tags_reject = ['bic','colorado','gocodecolorado','gis']
titles = list(sorted(dfCim.keys()))

for title in titles:
    dct=dfCim[title]['resource']
    cls=dfCim[title]['classification']
    if 'domain_category' not in cls:
        cls['domain_category']=[]
    if 'domain_tags' not in cls:
        cls['domain_tags']=[]
    else: 
        cls['domain_tags'] = [tag for tag in cls['domain_tags'] if tag not in tags_reject]
    domains[title] = cls['domain_category'] 
    tags[title] = cls['domain_tags']
    if 'domain_tags' not in cls:
        cls['domain_tags']=[]
    domains[title] = cls['domain_category'] 
    sentences[title] = (
        dct["name"] +' ' +
        dct["description"] + ' ' 
    ).lower()

    words[title] = (
        dct["name"] +' ' +
        dct["description"] + ' ' 
        ' '.join(cls['domain_tags']) + ' ' +
        ' '.join(dct['columns_description'])
    ).lower()
  #  cimString[title]=string 


words = [words[k] for k in titles]
sentences = [sentences[k] for k in titles]

In [54]:
# model = SentenceTransformer("all-MiniLM-L6-v2")
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
sem_emb = model.encode(sentences, convert_to_tensor=True)
sem_sim = util.cos_sim(sem_emb, sem_emb)

kw = TfidfVectorizer(stop_words="english")
kw_vecs = kw.fit_transform(words)
kw_sim = cosine_similarity(kw_vecs)

alpha = .0  # semantic weight
beta = 1.5 # keyword weight

combined_sim = (alpha * sem_sim) + (beta * kw_sim)
# --- Define a similarity threshold ---
threshold = 0.1  # tune this; 0.3–0.4 usually gives useful clusters
combined_sim = combined_sim.cpu()
rows = []
titles = list(dfCim.keys())
for i,title in enumerate(titles):
    for j,title2 in enumerate(titles[i+1:]):
        score = combined_sim[i, j]
        if score >= threshold:
            rows.append({
                "subject": title,
                "predicate": "related_to",
                "object": title2,
                "is_literal": False,
                "score":score
            })

# --- Write to Neo4j import file ---
output = pd.DataFrame(rows)
print(threshold,len(output))
output.to_csv("neo4j_related.csv", index=False)
print(f"✅ Created {len(output)} related edges for Neo4j import")


/tmp/ipykernel_2997/669397446.py:13: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  combined_sim = (alpha * sem_sim) + (beta * kw_sim)


0.1 41956
✅ Created 41956 related edges for Neo4j import


## Final 

In [3]:
def getCimDatasets():
    cim_url_query = "data.colorado.gov"
    allDatasets = []
    cimDatasets = {}
    
    # Connect to the Socrata API
    with Socrata(cim_url_query, None) as client:
        datasets = client.datasets()
        for dataset in datasets:
            allDatasets.append(dataset)
            if dataset['owner']['display_name'] == 'Colorado Information Marketplace' or dataset['owner']['display_name'] == "Business Intelligence Center of CO":
                title=dataset["resource"]["name"]
                w4x4=dataset["resource"]["id"]

                cimDatasets[title]=dataset
    
    return cimDatasets
dfCim = getCimDatasets()

In [4]:
DOMAIN_STOPWORDS = {
    "colorado","denver","county","city","state",
    "dola","local","affairs","department","office",
    "public","state","government","cdot","cogcc",

    "american","community","survey",
    "subset","demographics","estimate","estimates",
    "level","geography","geographic","tract","block",
    "boundary","boundaries","map","mapping","gis",

    "data","dataset","information","record","records",
    "table","source","provided","includes","including",
    "details","statistics","annual","monthly","daily",
    "historical","current","system","bic","gocodecolorado","gocode","go"    
}

In [5]:
sentences={}
words={}
domains = {}
tags = {}
tags_reject = ['bic','colorado','gocodecolorado','gis']
titles = list(sorted(dfCim.keys()))

for title in titles:
   
    dct=dfCim[title]['resource']
    cls=dfCim[title]['classification']

    if 'domain_category' not in cls:
        cls['domain_category']=' '
    else:
        if isinstance(cls['domain_category'],list):
           cls['domain_category'] = ' '.join([tag+' ' for tag in cls['domain_category'] if tag not in DOMAIN_STOPWORDS])

    if 'domain_tags' not in cls:
        cls['domain_tags']= " "
    else: 
        if isinstance(cls['domain_tags'],list):
           cls['domain_tags'] = ' '.join([tag+' ' for tag in cls['domain_tags'] if tag not in DOMAIN_STOPWORDS])

    if 'columns_description' not in dct:
        dct['columns_description']= " "
    else: 
        if isinstance(dct['columns_description'],list):
           dct['columns_description'] = ' '.join([tag+' ' for tag in dct['columns_description'] if tag not in tags_reject])
    
      
    domains[title] = cls['domain_category'] 
    tags[title] = cls['domain_tags']

    domains[title] = cls['domain_category'] 
    
    all_words = ' '.join([word.lower() for word in cls['domain_tags'].split() if word.lower() not in ENGLISH_STOP_WORDS])
    desc_words = ' '.join([word.lower() for word in dct['columns_description'].split() if word.lower() not in ENGLISH_STOP_WORDS])
   # words[title] = cls['domain_category'] + ' '

    words[title] = f"{dct['name'].lower()} {dct['description'].lower()} {cls['domain_category']} {all_words}".lower()
    #     # dct["name"] +' ' +
    #     # dct["description"] + ' ' 
    #   #  ' '.join(cls['domain_tags']) + ' ' +
    
  #  cimString[title]=string 


words = [words[k] for k in titles]

In [6]:
def debug_pair(i, j, words, matrix, cosine_sim, feature_names, top_terms=20):
    vec_i = matrix[i]       # sparse row
    vec_j = matrix[j]

    # Non-zero term indices
    nz_i = set(vec_i.indices)
    nz_j = set(vec_j.indices)

    overlap_idx = nz_i & nz_j

    # Cosine score
    score = cosine_sim[i, j]

    print("\n====================================")
    print(f"PAIR {i} <-> {j}")
    print("------------------------------------")
    print("Text A:", words[i][:200])
    print("Text B:", words[j][:200])
    print(f"\nCosine similarity: {score:.4f}")

    # Show all overlapping terms with their weights
    overlap_terms = []
    for k in overlap_idx:
        term = feature_names[k]
        wi = vec_i[0, k]
        wj = vec_j[0, k]
        overlap_terms.append((term, wi, wj, wi + wj))

    # Sort by combined TF-IDF weight (most influential)
    overlap_terms.sort(key=lambda x: x[3], reverse=True)

    print("\nTop overlapping terms:")
    for term, wi, wj, total in overlap_terms[:top_terms]:
        print(f"  {term:<20} TFIDF_A={wi:.4f}  TFIDF_B={wj:.4f}  sum={total:.4f}")

    print("====================================\n")

In [17]:
# model = SentenceTransformer("all-MiniLM-L6-v2")

# --- Define a similarity threshold ---
threshold = 0.1  # tune this; 0.3–0.4 usually gives useful clusters
tfidf = TfidfVectorizer(stop_words="english")
matrix = tfidf.fit_transform(words)
cosine_sim = cosine_similarity(matrix) # shape: (n_datasets, n_datasets)
feature_names = tfidf.get_feature_names_out()
rows = []
total = []
nint=0
for i,title in enumerate(titles):
    w4x4 = dfCim[title]['resource']['id']
    for j in range(i+1, len(titles)):
        title2 = titles[j]  
        w4x42 = dfCim[title2]['resource']['id']
        score = cosine_sim[i, j]
        total.append(score)
        if title.lower().find("charit") > -1 and title2.lower().find("charit") > -1:
                print("Title ",score,title)
                print("Title 2",title2)
       #         debug_pair(i,j, words, matrix, cosine_sim, feature_names)
  
        if score >= threshold:
            nint+=1
            # if nint > 10:
            #     break
            if title.lower().find("charit") > -1 and title2.lower().find("charit") > -1:
                print("Title ",score,title)
                print("Title 2",title2)
                debug_pair(i,j, words, matrix, cosine_sim, feature_names)
  
            rows.append({
                "subject": title,
                "subject_w4x4": w4x4,
                "predicate": "related_to",
                "object": title2,
                "object_w4x4": w4x42,
                "is_literal": False,
                "score":score
            })

# --- Write to Neo4j import file ---
output = pd.DataFrame(rows)
print(threshold,len(output))
output.to_csv("neo4j_related.csv", index=False, quoting=csv.QUOTE_ALL)
print(f"✅ Created {len(output)} related edges for Neo4j import")


Title  0.22978599216541465 Activities of Charities Operating in Colorado
Title 2 Campaign Reports for Solicitation Notices to Charities in Colorado
Title  0.22978599216541465 Activities of Charities Operating in Colorado
Title 2 Campaign Reports for Solicitation Notices to Charities in Colorado

PAIR 3 <-> 42
------------------------------------
Text A: activities of charities operating in colorado charitable organizations’ activities for the current tax year as reported to the irs with form 990, provided by colorado department of state (cdos) since 
Text B: campaign reports for solicitation notices to charities in colorado recent modifications in the data transformation process may result in data changes.  additional details about the changes can be foun

Cosine similarity: 0.2298

Top overlapping terms:
  charities            TFIDF_A=0.2716  TFIDF_B=0.1795  sum=0.4512
  charity              TFIDF_A=0.1358  TFIDF_B=0.2693  sum=0.4051
  state                TFIDF_A=0.2572  TFIDF_B=0.11

In [64]:
from collections import Counter

# Split all sentences into words and count them
all_words = [word for sentence in words for word in sentence.split()]
word_counts = Counter(all_words)
# 

In [1]:
import keras
import tf_keras
import transformers
print('keras', keras.__version__)
print('tf_keras', tf_keras.__version__)
print('transformers', transformers.__version__)

c:\Users\scien\miniconda3\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


c:\Users\scien\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


keras 3.13.2
tf_keras 2.20.1
transformers 4.57.1


In [18]:
output.shape

(40445, 7)

In [19]:
output.head()

,subject,subject_w4x4,predicate,object,object_w4x4,is_literal,score
0,2019 Novel Coronavirus COVID-19 (2019-nCoV) Da...,rkuy-jxyz,related_to,Breckenridge COVID-19 Business Modifications,xwar-3geb,False,0.133846
1,2019 Novel Coronavirus COVID-19 (2019-nCoV) Da...,rkuy-jxyz,related_to,Businesses Issued Outdoor Expansion Permits in...,6ntv-x37b,False,0.102691
2,2019 Novel Coronavirus COVID-19 (2019-nCoV) Da...,rkuy-jxyz,related_to,COVID-19 Colorado Cases Dashboard,ffjg-bf34,False,0.477584
3,2019 Novel Coronavirus COVID-19 (2019-nCoV) Da...,rkuy-jxyz,related_to,COVID-19 Economic Injury Disaster Loans (EIDL)...,8ant-mn9d,False,0.109368
4,2019 Novel Coronavirus COVID-19 (2019-nCoV) Da...,rkuy-jxyz,related_to,COVID-19 Liquor License Modifications in Colorado,e7wz-djhw,False,0.122093


In [23]:
output.to_csv("neo4j_related.csv", index=False,quoting=csv.QUOTE_ALL)
